In [1]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.messages import HumanMessage, SystemMessage

from llms_course.utils.llms import get_structured_output_llm

We can ensure the LLM only outputs specific values/value ranges using structured output mode.

For example, we can bound the value range on numeric fields (int, float) by setting the Field parameters `le` (less than or equal), `lt` (strictly less than), `ge` (greater than or equal), or `gt` (strictly greater than).

We can also restrict the LLM's output to specific values using the `Literal` type.

However, while it is possible to restrict the output of the LLM to specific values/value ranges, the LLM can still hallucinate by outputting improbable or incorrect values within those bounds. For example, having an LLM predict how happy a sentence sounds from 1-10 might still result in the LLM assigning a low score to a happy sentence.

In [2]:
class CustomerSupportChatAnalysis(BaseModel):
    """
    The sentiment level, frustration level, and whether the message requires escalation to a supervisor.

    Set `requires_escalation` to True if:
    - The customer requests to speak to a supervisor.
    - The customer would like a refund.
    """
    sentiment: Literal["positive", "negative", "neutral"] = Field(description = "The sentiment level of the message.")
    frustration_level: int = Field(description = "The frustration level of the message.", ge = 1, le = 10)
    requires_escalation: bool = Field(description = "Whether the message requires escalation to a supervisor. A frustration leve greater than 5 requires escalation.")


In [3]:
def analyze_sentiment(message: str) -> CustomerSupportChatAnalysis:
    """
    Analyze the sentiment, frustration level, and whether the message requires escalation to a supervisor of a text using an LLM.

    Args:
        message (str): The message to analyze.

    Returns:
        (CustomerSupportChatAnalysis): The analysis of the message.
    """
    llm = get_structured_output_llm(CustomerSupportChatAnalysis)

    system_prompt = """# Context
You are provided with a message from a customer within a customer support chat.

# Instructions
- Analyze the message from the customer for its sentiment, frustration level, and whether it requires escalation to a supervisor.

# Output Format
- The sentiment level should be one of the following: positive, negative, neutral.
- The frustration level should be an integer between 1 and 10.
- The requires_escalation should be a boolean.
"""
# It is good practice to actually post the entire schema class into the system prompt of the LLM
    messages = [
        SystemMessage(content = system_prompt),
        HumanMessage(content = f"Analyze the following message from the customer: ```{message}```")
    ]

    response = llm.invoke(messages)
    analysis = f"""Sentiment: {response.sentiment}
Frustration Level: {response.frustration_level}
Requires Escalation: {response.requires_escalation}"""
    return analysis


In [4]:
message = "I'm having trouble with my account. I've been trying to log in for the past hour and it's not working."

analysis = analyze_sentiment(message)
print(analysis)

Sentiment: negative
Frustration Level: 6
Requires Escalation: True


In [5]:
message = "Thanks for your help!"

analysis = analyze_sentiment(message)
print(analysis)

Sentiment: positive
Frustration Level: 1
Requires Escalation: False


In [6]:
message = "I want my money back!"

analysis = analyze_sentiment(message)
print(analysis)

Sentiment: negative
Frustration Level: 7
Requires Escalation: True


In [7]:
message = "How can I register for an account?"

analysis = analyze_sentiment(message)
print(analysis)

Sentiment: neutral
Frustration Level: 1
Requires Escalation: False


In [8]:
message = "Can I please speak to a supervisor?"

analysis = analyze_sentiment(message)
print(analysis)

Sentiment: neutral
Frustration Level: 6
Requires Escalation: True
